<a href="https://colab.research.google.com/github/i2m16/files/blob/main/IrisNNExample_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split

In [ ]:
class Model(nn.Module):
    def __init__(self, in_features=4, h1=8, h2=9,out_features=3):
        super(Model, self).__init__()

        self.fc1 = nn.Linear(in_features, h1)
        self.fc2 = nn.Linear(h1, h2)
        self.out = nn.Linear(h2, out_features)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.out(x)

        return x

In [ ]:
torch.manual_seed(41)

model = Model()

import pandas as pd
import matplotlib.pyplot as plt


url = 'https://gist.githubusercontent.com/curran/a08a1080b88344b0c8a7/raw/0e7a9b0a5d22642a06d3d5b9bcbad9890c8ee534/iris.csv'
my_df = pd.read_csv(url)

my_df['species'] = my_df['species'].replace('setosa',0.0)
my_df['species'] = my_df['species'].replace('versicolor',1.0)
my_df['species'] = my_df['species'].replace('virginica',2.0)

X = my_df.drop('species', axis=1)
y = my_df['species']

X = X.values
y = y.values

/tmp/ipykernel_2715/1829140025.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  my_df['species'] = my_df['species'].replace('virginica',2.0)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=41)

X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)
y_train = torch.LongTensor(y_train)
y_test = torch.LongTensor(y_test)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

epochs = 10000
losses = []

for i in range(epochs):
    y_pred = model.forward(X_train)

    loss = criterion(y_pred, y_train)

    losses.append(loss.detach().numpy())

    if i % 10 == 0:
        print(f'Epoch: {i} and loss:{loss}')

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

Epoch: 0 and loss:1.1251550912857056
Epoch: 10 and loss:1.1095242500305176
Epoch: 20 and loss:1.0960756540298462
Epoch: 30 and loss:1.0845398902893066
Epoch: 40 and loss:1.073915719985962
Epoch: 50 and loss:1.0637413263320923
Epoch: 60 and loss:1.0538722276687622
Epoch: 70 and loss:1.0440350770950317
Epoch: 80 and loss:1.0337213277816772
Epoch: 90 and loss:1.0227365493774414
Epoch: 100 and loss:1.0096259117126465
Epoch: 110 and loss:0.9954349398612976
Epoch: 120 and loss:0.9828447103500366
Epoch: 130 and loss:0.9664683938026428
Epoch: 140 and loss:0.9485272169113159
Epoch: 150 and loss:0.9305612444877625
Epoch: 160 and loss:0.9112032651901245
Epoch: 170 and loss:0.8891754150390625
Epoch: 180 and loss:0.8652693629264832
Epoch: 190 and loss:0.8407701253890991
Epoch: 200 and loss:0.8157405853271484
Epoch: 210 and loss:0.7899296283721924
Epoch: 220 and loss:0.7633073925971985
Epoch: 230 and loss:0.7376409769058228
Epoch: 240 and loss:0.7132112383842468
Epoch: 250 and loss:0.690205037593841

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

#plt.plot(range(epochs), losses)
#plt.ylabel("loss/error")
#plt.xlabel("epoch")
#plt.show()

with torch.no_grad():
    y_eval = model.forward(X_test)
    loss = criterion(y_eval, y_test)
    print(loss)

correct = 0
with torch.no_grad():
    for i, data in enumerate(X_test):
        y_val = model.forward(data)

        if y_test[i] == 0:
            x = "Setosa"
        elif y_test[i] == 1:
            x = "Versicolor"
        else:
            x = "Virginica"

        print(f'{i+1}.) {str(y_val)} {x} {y_test[i]} {y_val.argmax().item()}')

        if y_test[i] == y_val.argmax().item():
            correct +=1

    print(f'We got {correct} correct!')
print(loss)

#check new iris
new_iris = torch.tensor([4.7, 3.2, 1.3, 0.2])
with torch.no_grad():
    print(model(new_iris))

newer_iris = torch.tensor([5.9, 3.0, 5.1, 1.0])
with torch.no_grad():
    print(model(newer_iris))

#Save the NN Model
torch.save(model.state_dict(), 'iris_model.pt')

new_model = Model()
new_model.load_state_dict(torch.load('iris_model.pt'))

print(new_model.eval())


iris_test_data1 = torch.tensor([5.0,	3.4,	1.5,	0.2]) #Setosa
with torch.no_grad():
    print(model(iris_test_data1))

iris_test_data2 = torch.tensor([6.7,	3.1,	4.4,	1.4]) #Versicolor
with torch.no_grad():
    print(model(iris_test_data2))

iris_test_data3 = torch.tensor([5.8,	2.7,	5.1,	1.9]) #Virginica
with torch.no_grad():
    print(model(iris_test_data3))

tensor(0.7351)
1.) tensor([-10.5457,  -0.7886,  19.8669]) Virginica 2 2
2.) tensor([-13.5923,  -5.4192,  29.8110]) Virginica 2 2
3.) tensor([-15.6662,  -5.4833,  33.3320]) Virginica 2 2
4.) tensor([ 13.7711,  26.9458, -20.4398]) Versicolor 1 1
5.) tensor([-13.1443,  -3.0087,  26.5303]) Virginica 2 2
6.) tensor([ 25.0486,  37.8384, -37.2901]) Versicolor 1 1
7.) tensor([-8.7488,  2.1117, 16.7083]) Virginica 2 2
8.) tensor([ 14.7090,  28.0266, -21.9793]) Versicolor 1 1
9.) tensor([-11.9365,  -1.3884,  22.8139]) Virginica 2 2
10.) tensor([-14.5429,  -5.8926,  31.8985]) Virginica 2 2
11.) tensor([-5.5000,  5.4900, 11.3190]) Virginica 2 2
12.) tensor([ 109.7184,   97.8908, -141.6470]) Setosa 0 0
13.) tensor([  99.5630,   88.6761, -128.4604]) Setosa 0 0
14.) tensor([ 29.8516,  38.6285, -42.5867]) Versicolor 1 1
15.) tensor([  95.8675,   87.4895, -124.4457]) Setosa 0 0
16.) tensor([-0.6186, 11.0575,  3.3250]) Virginica 2 1
17.) tensor([ 100.6592,   90.1159, -130.0647]) Setosa 0 0
18.) tensor([